# ACTv7 on NMED-T raw EEG — runs A / B / C-grid / C-ACTv7

Tests whether cleaning the raw NMED-T EEG with the Adaptive Chirplet Transform makes each song's beat easier to detect at Muse-like electrode sites than (A) the dataset authors' cleaning or (B) minimal cleanup. Follows the "Testing alternative preprocessing" section of the project README.

**What this notebook does, per participant**
1. Downloads the two raw recordings from the [NMED-T Stanford repository](https://purl.stanford.edu/jn859kj8079), finds the 10 song onsets from `DIN_1`, keeps channels E57, E100, E26, E2 + E15, then deletes the raw files (optional) to save disk.
2. Builds three versions of each song, all re-referenced to E15, at 125 Hz, 15–255 s → `(k, 30000)`:
   - **B**: minimal (re-reference → downsample → epoch → median removal)
   - **C-grid**: B + the notebook's fast grid-based matching-pursuit ACT (10 chirplets / 2 s window)
   - **C-ACTv7**: B + the GitLab package's `ACTv7` (`for_eeg_v7` preset, 10 chirplets / 2 s window)
3. Loads the authors' cleaned files for **A** and runs `tempo_check` (own tempo + mismatched-tempo control) on every version.

Everything is checkpointed to `work/`, so a stopped run resumes where it left off.

**Files that must sit next to this notebook** (from the shared `EEG_Audio_Decomp` Drive folder): `nmedt_pilot.py`, `tempoHz.mat`. Optional, for the envelope test at the end: `pilot_trf.py`, `pooled_cv.py`, `audio/song21.wav`.

**About the GPU.** The `ACTv7` code we inspected is NumPy/SciPy on the CPU (L-BFGS-B + Newton refinement); cell 3 checks the package for any GPU switch. The real speed-up is **CPU cores**: windows run in parallel with joblib. Measured on Colab: ~8.7 s per window at 10 chirplets on one core. Cell 6 times it on this machine and projects the total before anything long starts.

**Settings were fixed before seeing results** (cell 2). If you change any, record the change; every variant tried should be reported.

## 1. Setup

In [ ]:
%pip install -q h5py joblib scipy pandas numpy
!git clone -q https://gitlab.com/adaptive-chirplet-transform/adaptive-chirplet-transform.git act_repo || echo "repo already present"
!pip install -q -e act_repo || echo "pip install of act_repo failed; importing from the folder instead"

In [ ]:
import os, sys, time, glob, urllib.request, shutil
import numpy as np, pandas as pd, h5py
import scipy.io as sio
from scipy.signal import resample_poly
from scipy.stats import binomtest
from joblib import Parallel, delayed

ACT_REPO = os.path.abspath('act_repo')
os.environ['ACT_REPO'] = ACT_REPO
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + ACT_REPO + os.pathsep + os.environ.get('PYTHONPATH', '')
sys.path[:0] = [os.getcwd(), ACT_REPO]

for f in ['nmedt_pilot.py', 'tempoHz.mat']:
    assert os.path.exists(f), f'{f} missing: copy it from the EEG_Audio_Decomp folder next to this notebook'
from nmedt_pilot import load_song, muse_like, tempo_check, CHANNEL_SETS
import act as actpkg
print('cpus:', os.cpu_count())

## 2. Settings (fixed in advance)

In [ ]:
# Participants (raw IDs). Cleaned index i corresponds to RAW_IDS[i].
RAW_IDS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 23]
PARTICIPANTS = RAW_IDS            # run all 20; use e.g. RAW_IDS[:5] for a shorter job

# Channels. Columns in the raw X array are channel number - 1.
CHANNELS = ['E57', 'E100', 'E26', 'E2']      # = CHANNEL_SETS['muse4'] order
REF = 'E15'                                    # Fpz-like reference, same as muse_like
SETS = {'muse4': ['E57', 'E100', 'E26', 'E2'], 'mastoid2': ['E57', 'E100']}

# ACTv7 run C-ACTv7
RUN_ACT7       = True
ACT7_CHANNELS  = ['E57', 'E100']   # mastoid2 only, to halve the cost; use CHANNELS for all four
ACT7_ORDER     = 10                # chirplets per 2 s window (same K as the grid ACT)
ACT7_KW        = ()                # extra for_eeg_v7 kwargs as tuple of pairs, e.g. (('newton_refine', False),)
N_JOBS         = os.cpu_count()
CHUNK          = 8                 # windows per parallel task

# Disk
DELETE_RAW = True                  # delete each raw .mat after extracting the channels (~1.6 GB per participant)
STANFORD   = ['https://stacks.stanford.edu/file/jn859kj8079/',
              'https://stacks.stanford.edu/file/druid:jn859kj8079/']

os.makedirs('data/raw', exist_ok=True); os.makedirs('work/act7', exist_ok=True)
TRIGS = list(range(21, 31))
tempos = sio.loadmat('tempoHz.mat')['tempoHz'].ravel()
col = lambda name: int(name[1:]) - 1

## 3. Does the ACT package have a GPU option?

In [ ]:
# Prints any lines mentioning GPU libraries. If a device / use_gpu style kwarg shows up,
# add it to ACT7_KW above (and note it). No matches = CPU only; parallel cores do the work.
!grep -rn -i -E "cupy|torch|cuda|use_gpu|device=" act_repo/act | head -30 || true

## 4. Download, triggers and channel extraction

In [ ]:
def download(name, dest_dir='data/raw', min_mb=100):
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest) and os.path.getsize(dest) > min_mb * 2**20:
        return dest
    last = None
    for base in STANFORD:
        try:
            tmp = dest + '.part'
            with urllib.request.urlopen(base + name, timeout=60) as r, open(tmp, 'wb') as f:
                shutil.copyfileobj(r, f, length=16 * 2**20)
            os.replace(tmp, dest)
            if os.path.getsize(dest) > min_mb * 2**20:
                return dest
        except Exception as e:
            last = e
    raise RuntimeError(f'could not download {name} ({last}). Download it by hand from '
                       f'https://purl.stanford.edu/jn859kj8079 into {dest_dir}/')

def read_triggers(fh):
    def deref(ref):
        d = fh[ref][()]
        if d.dtype == np.uint16:
            return ''.join(chr(c) for c in d.ravel())
        return d.ravel()[0] if d.size == 1 else d.ravel()
    din = fh['DIN_1']
    return [[deref(din[i, j]) for j in range(din.shape[1])] for i in range(din.shape[0])]

def song_onsets(rows):
    on = {}
    for r in rows:
        code_, sample = r[0], r[1]
        if code_.startswith('DI') and not code_.startswith('DIN'):
            n = int(code_[2:])
            if 21 <= n <= 30:
                on[n] = int(sample)
    return on

PAD_END = 2000   # 2 s beyond 255 s
def extract_participant(pid):
    # Returns {trig: (4, 32125) float32}: E15-referenced, 125 Hz, 0-257 s from onset.
    out_path = f'work/p{pid:02d}_raw125.npz'
    if os.path.exists(out_path):
        d = np.load(out_path)
        return {t: d[f'song{t}'] for t in TRIGS}
    names = CHANNELS + [REF]
    cols = [col(n) for n in names]
    order = np.argsort(cols); sorted_cols = [cols[i] for i in order]; undo = np.argsort(order)
    songs = {}
    for rec in (1, 2):
        path = download(f'{pid:02d}_{rec}_raw.mat')
        with h5py.File(path, 'r') as fh:
            fs = float(fh['fs'][0, 0]); assert fs == 1000.0, fs
            n = fh['X'].shape[0]
            for trig, onset in song_onsets(read_triggers(fh)).items():
                end = onset + 255000 + PAD_END
                assert end <= n, f'p{pid} song {trig} too short'
                seg = fh['X'][onset:end, sorted_cols][:, undo].T          # (5, 257000)
                x = seg[:4] - seg[4][None]                                 # re-reference to E15
                songs[trig] = resample_poly(x, 1, 8, axis=1).astype(np.float32)   # 1000 -> 125 Hz
        if DELETE_RAW:
            os.remove(path)
    assert sorted(songs) == TRIGS, f'p{pid}: found songs {sorted(songs)}'
    np.savez(out_path, **{f'song{t}': v for t, v in songs.items()})
    return songs

def epoch(x):
    # (k, 32125) at 125 Hz from onset -> 15-255 s, median removed: (k, 30000)
    x = x[:, 15 * 125: 255 * 125]
    return x - np.median(x, axis=1, keepdims=True)

## 5. Grid ACT (run C-grid) and ACTv7 worker

In [ ]:
# Grid matching-pursuit ACT, exactly as used for the participant-02 results.
FS, WIN, HOP, K = 125, 250, 125, 10
TCS = np.arange(0, 2.0, 0.25); FCS = np.arange(0.5, 40.01, 0.5)
DURS = [0.15, 0.4, 1.0]; CHIRPS = [-3.0, 0.0, 3.0]
t = np.arange(WIN) / FS
atoms = []
for tc in TCS:
    for fc in FCS:
        for d in DURS:
            for c in CHIRPS:
                tt = t - tc
                g = np.exp(-0.5 * (tt / d) ** 2) * np.exp(2j * np.pi * (fc * tt + 0.5 * c * tt ** 2))
                atoms.append(g / np.linalg.norm(g))
D = np.array(atoms, dtype=np.complex64).T
Dc, DR, DI = D.conj(), D.real, D.imag
HANN = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(WIN) / WIN)).astype(np.float32)

def act_frames(R):
    R = R.astype(np.float32).copy(); rec = np.zeros_like(R)
    for _ in range(K):
        best = np.abs(R @ Dc).argmax(1)
        C, S = DR[:, best].T, DI[:, best].T
        cc, ss, cs = (C*C).sum(1), (S*S).sum(1), (C*S).sum(1)
        rc, rs = (R*C).sum(1), (R*S).sum(1)
        det = cc*ss - cs**2
        ok = (ss > 1e-6 * cc) & (det > 1e-6 * cc * ss)
        det_safe = np.where(ok, det, 1.0)
        a = np.where(ok, (rc*ss - rs*cs) / det_safe, rc / cc)
        b = np.where(ok, (rs*cc - rc*cs) / det_safe, 0.0)
        part = a[:, None] * C + b[:, None] * S
        R -= part; rec += part
    return rec, R

def act_grid(x):
    n_ch, L = x.shape
    starts = np.arange(0, L - WIN + 1, HOP)
    frames = np.stack([x[:, s:s + WIN] for s in starts], 1)
    frames = frames - frames.mean(2, keepdims=True)
    rec, _ = act_frames(frames.reshape(-1, WIN))
    rec = rec.reshape(n_ch, len(starts), WIN)
    out = np.zeros((n_ch, L), dtype=np.float32)
    for j, s in enumerate(starts):
        out[:, s:s + WIN] += rec[:, j] * HANN
    return out

In [ ]:
%%writefile act7_worker.py
"""Helpers for running ACTv7 window-by-window in parallel worker processes.

Imported by the notebook and by each joblib worker. Each worker builds its own
ACTv7 object once (cached) and reuses it for every window it is given.
"""
import os
import sys
import functools
import itertools
import numpy as np

FS = 125.0          # ACT runs on the 125 Hz signal
WIN = 250           # 2 s windows
HOP = 125           # 50 % overlap
HANN = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(WIN) / WIN))

_repo = os.environ.get('ACT_REPO', '/content/act_repo')
if _repo not in sys.path:
    sys.path.insert(0, _repo)


# ----------------------------------------------------------------------------
# ACTv7 object, one per process
# ----------------------------------------------------------------------------
@functools.lru_cache(maxsize=None)
def get_act(kw_items=()):
    import act as actpkg
    return actpkg.for_eeg_v7(WIN, fs=FS, **dict(kw_items))


# ----------------------------------------------------------------------------
# Turning ACTv7 atoms back into a signal
# ----------------------------------------------------------------------------
# ACTv7 returns chirplet parameters, not a reconstructed window, and its
# parameter units are not documented in what we could see. calibrate()
# tries each plausible unit convention, least-squares fits the atoms to the
# window, and keeps the convention whose explained energy matches the
# package's own certificate. If none matches, it stops with an error rather
# than silently using a wrong reconstruction.

def _candidate_specs():
    return list(itertools.product(
        ['log_samples', 'log_seconds'],                  # logDt units
        ['hz', 'cyc_per_sample', 'cyc_per_window'],      # fc units
        ['hz_per_s', 'cyc_per_sample2', 'cyc_per_window2'],  # c units
        [1.0, 0.5],                                      # chirp term: c*u^2 or 0.5*c*u^2
    ))


def atom_shape(atom, spec):
    dt_u, fc_u, c_u, c_mult = spec
    n = np.arange(WIN, dtype=np.float64)
    u = n - float(atom.tc)                                # tc in samples
    dt = np.exp(float(atom.logDt))
    if dt_u == 'log_seconds':
        dt = dt * FS
    fc = float(atom.fc)
    fc = {'hz': fc / FS, 'cyc_per_sample': fc, 'cyc_per_window': fc / WIN}[fc_u]
    c = float(atom.c)
    c = {'hz_per_s': c / FS ** 2, 'cyc_per_sample2': c, 'cyc_per_window2': c / WIN ** 2}[c_u]
    env = np.exp(-0.5 * (u / dt) ** 2)
    return env * np.exp(2j * np.pi * (fc * u + c_mult * c * u ** 2))


def ls_fit(w, atoms, spec):
    """Least-squares fit of the atoms' real and imaginary parts to window w."""
    if len(atoms) == 0:
        return np.zeros_like(w)
    cols = []
    for a in atoms:
        g = atom_shape(a, spec)
        cols += [g.real, g.imag]
    B = np.stack(cols, 1)
    coef, *_ = np.linalg.lstsq(B, w, rcond=None)
    return B @ coef


def explained(w, rec):
    return 1.0 - np.sum((w - rec) ** 2) / np.sum(w ** 2)


def _array_attr(obj, names):
    for n in names:
        v = getattr(obj, n, None)
        if isinstance(v, np.ndarray) and v.shape[-1] == WIN:
            return n, v
    return None, None


def calibrate(windows, order, kw_items=()):
    """Decide how to rebuild a window from ACTv7's output.

    Returns ('cert_recon', name) | ('cert_resid', name) | ('spec', spec).
    """
    act = get_act(kw_items)
    w = windows[0]
    atoms, cert = act.transform(w, order=order)

    name, arr = _array_attr(cert, ['reconstruction', 'recon', 'approximation', 'signal_hat', 'fit'])
    if name:
        print(f'using cert.{name} as the reconstruction')
        return ('cert_recon', name)
    name, arr = _array_attr(cert, ['residual', 'residue', 'final_residual'])
    if name:
        print(f'using window - cert.{name} as the reconstruction')
        return ('cert_resid', name)

    scores = []
    for w in windows:
        atoms, cert = act.transform(w, order=order)
        target = float(cert.energy_explained_fraction)
        errs = [abs(explained(w, ls_fit(w, atoms, s)) - target) for s in _candidate_specs()]
        scores.append(errs)
    scores = np.array(scores)
    worst = scores.max(0)                      # worst mismatch across test windows
    best = int(worst.argmin())
    spec = _candidate_specs()[best]
    print(f'best atom convention {spec}: max |explained - cert| = {worst[best]:.4f}')
    if worst[best] > 0.02:
        raise RuntimeError(
            'No atom convention reproduces ACTv7\'s own energy_explained_fraction '
            f'(best mismatch {worst[best]:.3f}). Find the package\'s reconstruction '
            'function (grep the repo for "reconstruct" / "synth") and use it instead.')
    return ('spec', spec)


def rebuild(w, atoms, cert, mode):
    kind, val = mode
    if kind == 'cert_recon':
        return np.asarray(getattr(cert, val), dtype=np.float64).real
    if kind == 'cert_resid':
        return w - np.asarray(getattr(cert, val), dtype=np.float64).real
    return ls_fit(w, atoms, val)


# ----------------------------------------------------------------------------
# Worker entry point
# ----------------------------------------------------------------------------
def run_chunk(windows, order, mode, kw_items=()):
    """windows: (n, WIN) mean-centred. Returns (recons (n, WIN), explained (n,))."""
    act = get_act(kw_items)
    rec = np.zeros_like(windows, dtype=np.float64)
    ex = np.zeros(len(windows))
    for i, w in enumerate(windows):
        atoms, cert = act.transform(w, order=order)
        rec[i] = rebuild(w, atoms, cert, mode)
        ex[i] = float(cert.energy_explained_fraction)
    return rec, ex


# ----------------------------------------------------------------------------
# Windowing shared with the grid ACT
# ----------------------------------------------------------------------------
def split_windows(x):
    """x: (channels, L) -> frames (channels, n_frames, WIN) mean-centred, starts."""
    n_ch, L = x.shape
    starts = np.arange(0, L - WIN + 1, HOP)
    frames = np.stack([x[:, s:s + WIN] for s in starts], 1).astype(np.float64)
    frames = frames - frames.mean(2, keepdims=True)
    return frames, starts


def overlap_add(rec, starts, L):
    """rec: (channels, n_frames, WIN) -> (channels, L)."""
    out = np.zeros((rec.shape[0], L))
    for j, s in enumerate(starts):
        out[:, s:s + WIN] += rec[:, j] * HANN
    return out

## 6. Calibrate ACTv7 reconstruction and estimate run time

In [ ]:
import importlib, act7_worker as aw
importlib.reload(aw)

# ACTv7 returns chirplet parameters, not a signal. Work out how to rebuild a window from them
# (checked against the package's own energy_explained_fraction), using real windows from the
# first participant.
first = extract_participant(PARTICIPANTS[0])
x = first[21][[CHANNELS.index(c) for c in ACT7_CHANNELS]]
frames, starts = aw.split_windows(x)
test = frames[0, 20:24]
MODE = aw.calibrate(test, ACT7_ORDER, ACT7_KW)

aw.get_act(ACT7_KW).transform(test[0], order=1)            # warm-up
t0 = time.time(); aw.run_chunk(test, ACT7_ORDER, MODE, ACT7_KW)
per_win = (time.time() - t0) / len(test)
n_win = len(PARTICIPANTS) * len(TRIGS) * len(ACT7_CHANNELS) * frames.shape[1]
print(f'{per_win:.2f} s per window on one core | {n_win:,} windows | '
      f'~{per_win * n_win / N_JOBS / 3600:.1f} h with {N_JOBS} parallel jobs')

## 7. Main loop: runs B, C-grid, C-ACTv7 for every participant (resumable)

In [ ]:
def act7_song(x):
    frames, starts = aw.split_windows(x)                       # (ch, nf, WIN)
    flat = frames.reshape(-1, aw.WIN)
    chunks = [flat[i:i + CHUNK] for i in range(0, len(flat), CHUNK)]
    res = Parallel(n_jobs=N_JOBS, backend='loky')(
        delayed(aw.run_chunk)(ch, ACT7_ORDER, MODE, ACT7_KW) for ch in chunks)
    rec = np.concatenate([r[0] for r in res]).reshape(frames.shape)
    ex = np.concatenate([r[1] for r in res])
    return aw.overlap_add(rec, starts, x.shape[1]).astype(np.float32), float(ex.mean())

log = []
for pid in PARTICIPANTS:
    t0 = time.time()
    songs = extract_participant(pid)
    for run, fn in [('B', lambda x: x), ('Cgrid', act_grid)]:
        p = f'work/p{pid:02d}_{run}.npy'
        if not os.path.exists(p):
            np.save(p, np.stack([epoch(fn(songs[t])) for t in TRIGS]))       # (10, 4, 30000)
    if RUN_ACT7:
        idx = [CHANNELS.index(c) for c in ACT7_CHANNELS]
        for trig in TRIGS:
            p = f'work/act7/p{pid:02d}_song{trig}.npy'
            if os.path.exists(p):
                continue
            ts = time.time()
            y, ex = act7_song(songs[trig][idx])
            np.save(p, epoch(y))                                              # (k7, 30000)
            log.append(dict(pid=pid, song=trig, explained=ex, seconds=time.time() - ts))
            pd.DataFrame(log).to_csv('work/act7/log.csv', mode='a',
                                     header=not os.path.exists('work/act7/log.csv'), index=False)
            log = []
            print(f'  p{pid:02d} song {trig}: ACTv7 explained {ex:.2f}, {time.time() - ts:.0f}s')
    print(f'participant {pid:02d} done in {(time.time() - t0) / 60:.1f} min')

## 8. Run A (authors' cleaned files) for the same participants

In [ ]:
idx_clean = [RAW_IDS.index(p) for p in PARTICIPANTS]
sigA = {}
for trig in TRIGS:
    cache = f'work/A_song{trig}.npy'
    if os.path.exists(cache):
        sigA[trig] = np.load(cache)[:, :, idx_clean]
        continue
    path = download(f'song{trig}_Imputed.mat', dest_dir='data', min_mb=100)
    x = load_song(path, trig)
    full = muse_like(x, CHANNEL_SETS['muse4'])               # (4, 30000, 20), E15-referenced
    np.save(cache, full)
    sigA[trig] = full[:, :, idx_clean]
    del x, full
    print('A: song', trig)

## 9. Tempo check: own tempo vs mismatched-tempo control

In [ ]:
def stack(run):
    # -> ({trig: (k, 30000, S)}, channel names)
    if run == 'C-ACTv7':
        arr = {t: np.stack([np.load(f'work/act7/p{p:02d}_song{t}.npy') for p in PARTICIPANTS], -1) for t in TRIGS}
        return arr, ACT7_CHANNELS
    if run == 'A':
        return sigA, CHANNELS
    name = {'B': 'B', 'C-grid': 'Cgrid'}[run]
    per = np.stack([np.load(f'work/p{p:02d}_{name}.npy') for p in PARTICIPANTS], -1)   # (10, 4, 30000, S)
    return {t: per[i] for i, t in enumerate(TRIGS)}, CHANNELS

def related(a, b, tol=0.05):
    mine = [a * m for m in (0.25, 0.5, 1, 2, 4)]
    theirs = [b * m for m in (0.5, 1, 2)]
    return any(abs(x - y) < tol for x in mine for y in theirs)

S = len(PARTICIPANTS)
runs = ['A', 'B', 'C-grid'] + (['C-ACTv7'] if RUN_ACT7 else [])
rows = []
for run in runs:
    sigs, names = stack(run)
    for set_name, chans in SETS.items():
        if not all(c in names for c in chans):
            continue
        ix = [names.index(c) for c in chans]
        own = mism = mism_n = 0
        for i, trig in enumerate(TRIGS):
            own += int(round(tempo_check(sigs[trig][ix], tempos[i])['frac_above_null95'] * S))
            for j in range(10):
                if j == i or related(tempos[i], tempos[j]):
                    continue
                mism += int(round(tempo_check(sigs[trig][ix], tempos[j])['frac_above_null95'] * S))
                mism_n += S
        n = 10 * S
        rows.append(dict(run=run, channels=set_name,
                         own=f'{own}/{n}', own_pct=round(100 * own / n, 1),
                         p_vs_5pct=binomtest(own, n, 0.05, alternative='greater').pvalue,
                         mismatched=f'{mism}/{mism_n}', mism_pct=round(100 * mism / mism_n, 1)))
res = pd.DataFrame(rows)
res.to_csv('work/tempo_ABC.csv', index=False)
print(res.to_string(index=False))

**Reading it.** `own_pct` = participant-songs whose own tempo beats the 95th-percentile null (chance ≈ 5%). `mism_pct` is the control and should stay near 5%; if a run raises both, it is creating peaks rather than revealing the beat. The same participants recur across songs, so the binomial p-value is optimistic (README caveat).

## 11. What to send back
- `work/tempo_ABC.csv` (and `work/pooled_cv_ABC.csv` if run)
- `work/act7/log.csv` (energy explained and seconds per song)
- the printed output of cells 3 and 6 (GPU check, reconstruction mode, timing)